In [1]:
import torch
from torch_geometric.data import HeteroData
import networkx as nx
from typing import Tuple, Dict, Any
from tqdm import tqdm
import copy

import os
import sys
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()
sys.path.append(os.path.dirname(base_dir))
from utils.graph_utils import load_graph, save_graph

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [79]:
adni_ecdf_hybrid = load_graph("./data/ADNI/G_adni_hybrid_ecdf.pkl")
geo_ecdf_hybrid = load_graph("./data/GEO/GSE33000_ad_hd/G_geo_hybrid_ecdf.pkl")
geo_logfc_hybrid = load_graph("./data/GEO/GSE33000_ad_hd/G_geo_hybrid_logfc.pkl")
ad_kg = load_graph("./data/KG/ad_network_with_reverse_edges.pkl")
healthy_kg = load_graph("./data/KG/healthy_aging_kg.pkl")

Loaded graph from ./data/ADNI/G_adni_hybrid_ecdf.pkl: 8275 nodes, 72920 edges
Loaded graph from ./data/GEO/GSE33000_ad_hd/G_geo_hybrid_ecdf.pkl: 8287 nodes, 158278 edges
Loaded graph from ./data/GEO/GSE33000_ad_hd/G_geo_hybrid_logfc.pkl: 8287 nodes, 228134 edges
Loaded graph from ./data/KG/ad_network_with_reverse_edges.pkl: 3732 nodes, 15712 edges
Loaded graph from ./data/KG/healthy_aging_kg.pkl: 4161 nodes, 7406 edges


In [26]:
causal_relations = {'HAS__ACTIVITY',
 'decreases',
 'directly_decreases',
 'directly_increases',
 'down_reg',
 'has__abundance',
 'has__complex',
 'has__fragment',
 'has__from_location',
 'has__gene',
 'has__location',
 'has__pmod',
 'has__products',
 'has__protein',
 'has__reactants',
 'has__rna',
 'has__to_location',
 'has__variant',
 'has_fragmented_protein',
 'has_located_abundance',
 'has_located_protein',
 'has_located_rna',
 'has_modified_protein',
 'has_variant_gene',
 'has_variant_protein',
 'has_variant_rna',
 'increases',
 'is_a',
 'regulates',
 'rev_decreases',
 'rev_directly_decreases',
 'rev_directly_increases',
 'rev_down_reg',
 'rev_increases',
 'rev_is_a',
 'rev_regulates',
 'rev_up_reg',
 'similar',
 'transcribed_to',
 'translated_to',
 'up_reg'}

In [72]:
def get_relation(u,v,rel, attr):
    if len(str(rel)) < 30:
        # get ad_kg relations
        relation = rel if isinstance(rel, str) else None
    else:
        # get healthy_kg relations
        relation = attr.get('type')
    if not relation:
        # get patient_protein relations
        relation = attr.get('relation')
    
    return relation

def process_kg_for_gnn(kg: nx.MultiDiGraph, causal_keywords: list|set):
    """
    Cleans a KG by removing non-causal relations and optionally adding reverse edges.
    
    Args:
        kg: The input MultiDiGraph.
        causal_keywords: List of strings that must be present in the 'relation' 
                         attribute to keep the edge.
    """
    
    # 1. Collect all edge types (for reporting/inspection)
    all_rel_types = set()
    for u, v, rel, attr in kg.edges(data=True, keys=True):
        relation = get_relation(u,v,rel,attr)
        all_rel_types.add(relation)
    print(f"Found {len(all_rel_types)} initial unique relations: {all_rel_types}")

    # 2. Create Cleaned KG (Causal Only)
    cleaned_kg = nx.MultiDiGraph()
    cleaned_kg.add_nodes_from(kg.nodes(data=True))
    
    removed_count = 0
    for u, v, rel, data in kg.edges(data=True, keys=True):
        relation = str(get_relation(u,v,rel,data)).lower()
        # Keep only if a causal keyword is found in the relation string
        if any(key in relation for key in causal_keywords):
            cleaned_kg.add_edge(u, v, relation, **data)
        else:
            # print(relation)
            removed_count += 1
            
    print(f"Removed {removed_count} non-causal edges.")

    # 3. Create Reversed KG
    # We copy the cleaned one so the reversed one is also 'causal-only'
    reversed_kg = copy.deepcopy(cleaned_kg)

    added_rev_count = 0
    # Use list() to avoid "dictionary changed size during iteration" error
    edges_to_process = list(cleaned_kg.edges(data=True, keys=True))

    for u, v, rel, data in tqdm(edges_to_process, desc="Checking/Adding reverse edges"):
        relation = get_relation(u,v,rel,data)
        # skip patient-patient
        if relation == 'similar':
            continue
        if 'rev' in relation:
            continue
        else:
            rev_rel = f"rev_{relation}"
        
        # Check if a reverse edge already exists
        # In a MultiDiGraph, we check all edges between v and u
        has_rev = False
        if reversed_kg.has_edge(v, u):
            key = reversed_kg[v][u]
            edge_type = list(key.keys())[0]
            #print(edge_type)
            if edge_type == rev_rel: has_rev = True
                    
        if not has_rev:
            # Add the reverse edge with the same attributes but flipped nodes
            rev_data = copy.deepcopy(data)
            #print(f"Add reverse edge {rev_rel} to KG")
            reversed_kg.add_edge(v, u, rev_rel, **rev_data)
            added_rev_count += 1

    print(f"Added {added_rev_count} reverse edges.")
    
    return cleaned_kg, reversed_kg


In [ ]:
cleaned_adkg, reversed_adkg = process_kg_for_gnn(ad_kg, causal_relations)
#save_graph(cleaned_adkg, "./data/KG/ad_kg_noncausal_removed.pkl")

Found 20 initial unique relations: {'rev_negative_correlation', 'rev_regulates', 'rev_decreases', 'negative_correlation', 'positive_correlation', 'directly_decreases', 'association', 'increases', 'directly_increases', 'decreases', 'rev_positive_correlation', 'rev_increases', 'regulates', 'causes_no_change', 'rev_is_a', 'rev_association', 'rev_directly_increases', 'rev_directly_decreases', 'is_a', 'rev_causes_no_change'}
Removed 5158 non-causal edges.


Checking/Adding reverse edges: 100%|██████████| 10554/10554 [00:00<00:00, 528860.53it/s]

Added 125 reverse edges.
Saved graph to ./data/KG/ad_kg_noncausal_removed.pkl: 3732 nodes, 10554 edges
Saved graph to ./data/KG/ad_kg_with_reversed_edges.pkl: 3732 nodes, 10554 edges


In [78]:
cleaned_healthkg, reversed_healthkg = process_kg_for_gnn(healthy_kg, causal_relations)

Found 31 initial unique relations: {'negative_correlation', 'positive_correlation', 'has__products', 'has__fragment', 'has_modified_protein', 'has__complex', 'has__location', 'has__abundance', 'has__variant', 'increases', 'association', 'transcribed_to', 'has__to_location', 'translated_to', 'has_variant_gene', 'has_fragmented_protein', 'has_located_protein', 'has_variant_rna', 'decreases', 'regulates', 'has_located_abundance', 'HAS__ACTIVITY', 'has__rna', 'has__pmod', 'has__protein', 'has__reactants', 'has_located_rna', 'has_variant_protein', 'is_a', 'has__from_location', 'has__gene'}
Removed 63 non-causal edges.


Checking/Adding reverse edges: 100%|██████████| 6896/6896 [00:00<00:00, 89752.25it/s]

Added 6896 reverse edges.


process adni graph

In [101]:
cleaned_adni_ecdf, reversed_adni_ecdf = process_kg_for_gnn(adni_ecdf_hybrid, causal_relations)

Found 49 initial unique relations: {'rev_negative_correlation', 'rev_regulates', 'rev_decreases', 'negative_correlation', 'positive_correlation', 'has__products', 'directly_decreases', 'similar', 'has__fragment', 'up_reg', 'has_modified_protein', 'has__complex', 'has__location', 'down_reg', 'rev_down_reg', 'has__abundance', 'has__variant', 'transcribed_to', 'increases', 'association', 'has__to_location', 'translated_to', 'has_variant_gene', 'has_fragmented_protein', 'has_located_protein', 'has_variant_rna', 'directly_increases', 'decreases', 'rev_positive_correlation', 'rev_increases', 'regulates', 'has_located_abundance', 'HAS__ACTIVITY', 'has__rna', 'causes_no_change', 'rev_is_a', 'has__protein', 'has__pmod', 'has__reactants', 'rev_up_reg', 'rev_association', 'rev_directly_increases', 'rev_directly_decreases', 'has_located_rna', 'has_variant_protein', 'is_a', 'rev_causes_no_change', 'has__from_location', 'has__gene'}
Removed 5221 non-causal edges.


Checking/Adding reverse edges: 100%|██████████| 64886/64886 [00:00<00:00, 383680.13it/s]

Added 7019 reverse edges.


process geo graph

In [80]:
cleaned_geo_ecdf, reversed_geo_ecdf = process_kg_for_gnn(geo_ecdf_hybrid, causal_relations)
cleaned_geo_logfc, reversed_geo_logfc = process_kg_for_gnn(geo_logfc_hybrid, causal_relations)

Found 49 initial unique relations: {'rev_negative_correlation', 'rev_regulates', 'rev_decreases', 'negative_correlation', 'positive_correlation', 'has__products', 'directly_decreases', 'similar', 'has__fragment', 'up_reg', 'has_modified_protein', 'has__complex', 'has__location', 'down_reg', 'rev_down_reg', 'has__abundance', 'has__variant', 'transcribed_to', 'increases', 'association', 'has__to_location', 'translated_to', 'has_variant_gene', 'has_fragmented_protein', 'has_located_protein', 'has_variant_rna', 'directly_increases', 'decreases', 'rev_positive_correlation', 'rev_increases', 'regulates', 'has_located_abundance', 'HAS__ACTIVITY', 'has__rna', 'causes_no_change', 'rev_is_a', 'has__protein', 'has__pmod', 'has__reactants', 'rev_up_reg', 'rev_association', 'rev_directly_increases', 'rev_directly_decreases', 'has_located_rna', 'has_variant_protein', 'is_a', 'rev_causes_no_change', 'has__from_location', 'has__gene'}
Removed 5221 non-causal edges.


Checking/Adding reverse edges: 100%|██████████| 148808/148808 [00:00<00:00, 538215.10it/s]


Added 7019 reverse edges.
Found 49 initial unique relations: {'rev_negative_correlation', 'rev_regulates', 'rev_decreases', 'negative_correlation', 'positive_correlation', 'has__products', 'directly_decreases', 'similar', 'has__fragment', 'up_reg', 'has_modified_protein', 'has__complex', 'has__location', 'down_reg', 'rev_down_reg', 'has__abundance', 'has__variant', 'transcribed_to', 'increases', 'association', 'has__to_location', 'translated_to', 'has_variant_gene', 'has_fragmented_protein', 'has_located_protein', 'has_variant_rna', 'directly_increases', 'decreases', 'rev_positive_correlation', 'rev_increases', 'regulates', 'has_located_abundance', 'HAS__ACTIVITY', 'has__rna', 'causes_no_change', 'rev_is_a', 'has__protein', 'has__pmod', 'has__reactants', 'rev_up_reg', 'rev_association', 'rev_directly_increases', 'rev_directly_decreases', 'has_located_rna', 'has_variant_protein', 'is_a', 'rev_causes_no_change', 'has__from_location', 'has__gene'}
Removed 5221 non-causal edges.


Checking/Adding reverse edges: 100%|██████████| 218664/218664 [00:00<00:00, 550907.62it/s] 

Added 7019 reverse edges.


In [ ]:
def normalize_and_deduplicate_kg(kg: nx.MultiDiGraph):
    """
    1. Normalizes 'directly_increases' -> 'increases'
    2. Merges duplicate edges between the same nodes with the same relation.
    """
    # Define the mapping for normalization
    normalization_map = {
        'directly_increases': 'increases',
        'directly_decreases': 'decreases'
    }

    # Create a new graph to store the deduplicated result
    # We use a standard DiGraph first to handle deduplication naturally, 
    # then convert back to MultiDiGraph if needed.
    new_kg = nx.MultiDiGraph()
    new_kg.add_nodes_from(kg.nodes(data=True))

    # Track seen edges: (u, v, relation) -> max_weight or shared_attrs
    seen_edges = {}

    for u, v, data in tqdm(kg.edges(data=True), desc="Normalizing and Deduplicating"):
        # 1. Normalize the relation name
        rel = data.get('relation', 'unknown')
        norm_rel = normalization_map.get(rel, rel)
        
        # 2. Create a unique key for the edge
        edge_key = (u, v, norm_rel)
        
        # 3. Logic for merging attributes (e.g., keep the highest weight)
        current_weight = float(data.get('weight', 1.0))
        
        if edge_key not in seen_edges:
            # First time seeing this connection
            edge_data = data.copy()
            edge_data['relation'] = norm_rel
            seen_edges[edge_key] = edge_data
        else:
            # Duplicate found! Merge logic: keep the maximum weight
            existing_weight = float(seen_edges[edge_key].get('weight', 1.0))
            if current_weight > existing_weight:
                seen_edges[edge_key]['weight'] = current_weight

    # 4. Add the unique edges to the new graph
    for (u, v, rel), final_data in seen_edges.items():
        new_kg.add_edge(u, v, **final_data)

    print(f"Original edges: {kg.number_of_edges()}")
    print(f"Deduplicated edges: {new_kg.number_of_edges()}")
    
    return new_kg

In [98]:
import re

def sanitize_node_types(G):
    """
    Standardizes node types while preserving existing PascalCase names.
    - 'Cell_surface_expression' -> 'CellSurfaceExpression'
    - 'CellSurfaceExpression' -> 'CellSurfaceExpression' (UNTOUCHED)
    - 'biological_process' -> 'BiologicalProcess'
    """
    def fix_type_name(text):
        if not text:
            return "Unknown"
        
        # If there are underscores or spaces, we need to join them
        if '_' in text or ' ' in text:
            words = re.split(r'[_\s]+', str(text))
            # Capitalize each part and join: 'cell_surface' -> 'CellSurface'
            return ''.join(word[0].upper() + word[1:] if len(word) > 0 else '' for word in words)
        
        # If no delimiters, just ensure the very first letter is uppercase
        # but leave the rest of the string exactly as it is.
        return text[0].upper() + text[1:]

    type_changes = {}
    
    for node, attrs in G.nodes(data=True):
        old_type = attrs.get('type')
        if old_type:
            new_type = fix_type_name(old_type)
            if old_type != new_type:
                type_changes[old_type] = new_type
            attrs['type'] = new_type
            
    if type_changes:
        print("Sanitized Node Types (Smart Mapping):")
        for old, new in sorted(type_changes.items()):
            print(f"  {old} -> {new}")
            
    return G

In [103]:
reversed_adni_ecdf = sanitize_node_types(reversed_adni_ecdf)
reversed_geo_ecdf = sanitize_node_types(reversed_geo_ecdf)
reversed_geo_logfc = sanitize_node_types(reversed_geo_logfc)

Sanitized Node Types (Smart Mapping):
  Biological_process -> BiologicalProcess
  Cell_secretion -> CellSecretion
  Cell_surface_expression -> CellSurfaceExpression
  From_location -> FromLocation
  Fusion_protein -> FusionProtein
  To_location -> ToLocation
Sanitized Node Types (Smart Mapping):
  Biological_process -> BiologicalProcess
  Cell_secretion -> CellSecretion
  Cell_surface_expression -> CellSurfaceExpression
  From_location -> FromLocation
  Fusion_protein -> FusionProtein
  To_location -> ToLocation


In [104]:
import pandas as pd
import numpy as np

def process_and_inject_features(G, exp_df):
    """
    1. Normalizes the expression dataframe.
    2. Updates the 'x' attribute for all Patient nodes in the graph.
    """
    print("Normalizing patient features...")
    
    # --- Step A: Normalization ---
    # Remove genes with no variation
    df_clean = exp_df.loc[:, exp_df.std() > 0]
    # Z-score: (x - mean) / std
    df_norm = (df_clean - df_clean.mean()) / df_clean.std()
    # Fill any remaining NaNs (from the normalization math) with 0 (the mean)
    df_norm = df_norm.fillna(0)
    
    # --- Step B: Injection ---
    updated_count = 0
    patient_nodes = [n for n, d in G.nodes(data=True) if d.get('type') == 'Patient']
    
    for node in patient_nodes:
        if node in df_norm.index:
            # Update the 'x' attribute with the normalized vector
            G.nodes[node]['x'] = df_norm.loc[node].values.astype(np.float32)
            updated_count += 1
        else:
            # Safety: If a patient is in the graph but not the expression file,
            # we need to ensure they have a zero-vector of the correct length.
            feature_dim = df_norm.shape[1]
            G.nodes[node]['x'] = np.zeros(feature_dim, dtype=np.float32)
            print(f"Warning: Patient {node} not found in expression data. Initializing with zeros.")

    print(f"Successfully updated features for {updated_count} patients.")
    return G, df_norm.shape[1] # Return graph and the new feature dimension

In [107]:
df_adni = pd.read_csv("./data/ADNI/adni_exp_2cls.csv", index_col=0)
df_geo = pd.read_csv("./data/GEO/GSE33000_ad_hd/GSE33000_exp_2cls.csv", index_col=0).T
df_adni

,A1BG,A1CF,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AACSP1,...,ZW10,ZWILCH,ZWINT,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
116_S_1249,3.651,2.2865,3.039,2.3395,2.783,2.25300,2.081,7.043,3.30950,2.202,...,5.748,3.77150,4.0365,4.353000,5.1763,1.9045,5.31750,9.2190,7.3010,5.7490
037_S_4410,3.183,2.1230,3.543,2.2085,2.383,2.37550,1.733,6.773,3.27625,2.317,...,5.974,4.17300,4.4415,4.520667,5.0964,1.9265,5.38800,8.3785,6.7580,6.0935
006_S_4153,3.278,2.3545,3.528,2.1745,2.593,2.46825,1.841,6.910,3.20875,2.540,...,5.119,3.91275,4.6210,4.230667,5.1143,2.2315,5.62800,9.1085,7.3365,5.2615
116_S_1232,3.371,2.3725,3.835,2.1545,2.570,2.51925,2.249,7.209,3.24950,2.559,...,4.904,3.73800,4.4435,4.050667,5.1520,2.0545,5.46300,9.3210,7.1685,4.7340
128_S_0205,3.194,2.3560,3.146,2.1230,2.673,2.52150,1.766,7.079,3.10525,2.448,...,5.571,4.07650,4.4430,4.366667,5.2429,1.9105,5.56125,8.5345,6.9175,5.6455
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
014_S_4668,3.330,2.2820,3.497,2.2965,3.155,2.67350,2.158,6.950,3.33150,2.923,...,4.309,3.60500,4.2075,3.903667,4.9955,2.1810,5.41125,9.3105,6.7660,5.0440
130_S_0289,3.368,2.2950,3.128,1.9650,2.622,2.45225,1.953,6.878,3.10100,2.481,...,5.649,3.86475,4.6765,4.663333,5.1113,2.0300,5.36250,9.1965,7.1770,5.5510
009_S_2381,3.302,2.5075,3.524,2.2525,2.876,2.76275,2.089,6.805,3.09150,2.650,...,4.523,3.50150,4.4685,3.962333,5.0892,1.9320,5.44275,9.2900,6.7035,4.7915
041_S_4014,3.532,2.4545,3.609,2.3495,3.081,2.69125,2.024,7.257,3.11425,2.497,...,4.037,3.68525,4.0480,3.976333,4.8241,2.0075,5.05725,9.5810,6.6865,3.8660


In [110]:
df_geo

,A1BG,A2M,A2ML1,A3GALT2,A4GALT,A4GNT,AAAS,AACS,AADAC,AADACL1,...,ZXDB,ZXDC,ZYG11B,ZYG11BL,ZYX,ZZEF1,ZZZ3,tcag7.1017,tcag7.216,tcag7.981
GSM1423780,-0.066694,0.023919,-0.031845,0.015204,0.089450,-0.167457,0.023215,0.004379,0.041729,0.074247,...,0.066588,-0.020308,0.080426,-0.057932,0.007106,0.106203,0.063151,0.066667,0.060402,0.023945
GSM1423781,0.054097,-0.012131,-0.030740,0.034282,-0.006780,0.068906,-0.004402,-0.029354,0.003009,-0.035774,...,0.035217,-0.000390,-0.027666,0.033120,0.065091,-0.040029,-0.118265,0.096145,-0.007896,0.022231
GSM1423782,0.025282,0.001929,-0.085577,-0.023002,-0.314342,0.092161,-0.043387,0.069851,0.010428,0.116397,...,0.072900,-0.042220,0.085496,-0.020298,0.040479,-0.146334,-0.027519,0.165024,-0.008110,0.118489
GSM1423783,0.002485,0.034625,-0.066813,-0.018289,0.094836,0.073653,0.073933,-0.043867,0.045298,-0.087155,...,-0.064955,-0.034116,-0.109619,0.029910,0.079794,-0.000814,-0.250529,-0.014006,-0.238408,-0.042106
GSM1423784,-0.092606,0.019591,-0.030884,0.020118,-0.206281,0.050492,-0.003392,0.055446,0.111875,0.152810,...,-0.091836,-0.050365,0.105040,0.052845,-0.013048,0.023013,0.051360,0.054292,0.115933,0.035485
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM1424242,-0.035529,-0.067985,-0.115346,-0.045191,-0.085769,-0.041509,-0.018460,0.108033,0.010502,0.153964,...,-0.017694,-0.130394,0.012196,-0.022353,0.032304,-0.100798,0.016300,0.053928,-0.060836,0.176064
GSM1424243,0.075622,-0.010792,-0.088835,-0.021300,-0.096128,-0.037948,0.038506,0.107594,0.001454,0.062382,...,0.025705,-0.133761,-0.092569,0.121448,0.159210,0.017417,-0.108027,0.072586,0.090821,0.010657
GSM1424244,0.065244,-0.087919,-0.120331,-0.038014,0.068436,-0.003720,0.013911,0.034820,0.105181,0.085700,...,-0.043359,-0.048832,-0.040329,0.001067,0.012171,0.130757,0.016830,0.094532,-0.294262,0.086503
GSM1424245,0.054953,0.072740,-0.094495,-0.028996,-0.039163,-0.066074,0.028088,0.070063,-0.028701,0.052751,...,-0.040056,-0.091335,-0.080240,0.045302,0.084873,0.083839,-0.073553,0.106768,-0.118929,0.061755


In [ ]:
reversed_adni_ecdf, _ = process_and_inject_features(reversed_adni_ecdf, df_adni)
reversed_geo_ecdf, _ = process_and_inject_features(reversed_geo_ecdf, df_geo)
reversed_geo_logfc, _ = process_and_inject_features(reversed_geo_logfc,df_geo)

Normalizing patient features...
Successfully updated features for 455 patients.
Normalizing patient features...
Successfully updated features for 467 patients.
Normalizing patient features...
Successfully updated features for 467 patients.


In [114]:
# save graphs
save_graph(reversed_geo_ecdf, './data/GEO/GSE33000_ad_hd/G_geo_ReversedHybrid_ecdf.pkl')
save_graph(reversed_geo_logfc, './data/GEO/GSE33000_ad_hd/G_geo_ReversedHybrid_logfc.pkl')
save_graph(reversed_adni_ecdf, "./data/ADNI/G_adni_ReversedHybrid_ecdf.pkl")

Saved graph to ./data/GEO/GSE33000_ad_hd/G_geo_ReversedHybrid_ecdf.pkl: 8287 nodes, 155702 edges
Saved graph to ./data/GEO/GSE33000_ad_hd/G_geo_ReversedHybrid_logfc.pkl: 8287 nodes, 225558 edges
Saved graph to ./data/ADNI/G_adni_ReversedHybrid_ecdf.pkl: 8275 nodes, 71780 edges


### Networkx to HeteroData

In [83]:
def networkx_to_hetero_data(graph: nx.MultiDiGraph) -> HeteroData:
    """Converts a NetworkX graph to a PyTorch Geometric HeteroData object."""
    print("Starting conversion from NetworkX to HeteroData...")
    data = HeteroData()
    node_mappings: Dict[str, Dict[Any, int]] = {}

    for node_id, attrs in graph.nodes(data=True):
        node_type = attrs.get('type')
        if node_type not in node_mappings:
            node_mappings[node_type] = {}
        if node_id not in node_mappings[node_type]:
            node_mappings[node_type][node_id] = len(node_mappings[node_type])

    for node_type, mapping in node_mappings.items():
        data[node_type].num_nodes = len(mapping)
    print(f"Found {len(node_mappings)} node types.")

    edge_lists: Dict[tuple, list] = {}
    for u, v, rel_type in graph.edges(keys=True):
        source_type, dest_type = graph.nodes[u]['type'], graph.nodes[v]['type']
        edge_type_tuple = (source_type, str(rel_type), dest_type)
        if edge_type_tuple not in edge_lists:
            edge_lists[edge_type_tuple] = []
        src_idx, dest_idx = node_mappings[source_type][u], node_mappings[dest_type][v]
        edge_lists[edge_type_tuple].append([src_idx, dest_idx])

    for edge_type_tuple, edges in edge_lists.items():
        data[edge_type_tuple].edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    print(f"Found {len(edge_lists)} edge types.")
    print("Conversion complete!")
    return data, node_mappings

In [89]:
def convert_to_hetero_data(G: nx.MultiDiGraph):
    """
    Converts the hybrid Patient-Protein network into a PyTorch Geometric HeteroData object.
    Initializes features for ALL node types to match Patient feature dimensions.
    """
    print("Starting conversion from NetworkX to HeteroData...")
    data = HeteroData()
    node_mappings = {} # {node_type: {node_name: integer_index}}
    
    # 1. Categorize Nodes and Map Indices
    for node, attrs in G.nodes(data=True):
        n_type = attrs.get('type')
        
        if n_type not in node_mappings:
            node_mappings[n_type] = {}
        
        if node not in node_mappings[n_type]:
            node_mappings[n_type][node] = len(node_mappings[n_type])

    # 2. Process Patient Data (The "Reference" Features)
    p_map = node_mappings.get('Patient', {})
    if not p_map:
        raise ValueError("No 'Patient' nodes found in the graph to use as a feature dimension reference.")
    
    p_ids = sorted(p_map, key=p_map.get)
    patient_x = torch.tensor([G.nodes[pid]['x'] for pid in p_ids], dtype=torch.float)
    
    # Capture the dimension D (e.g., number of genes/features)
    feature_dim = patient_x.size(1) 
    
    data['Patient'].x = patient_x
    data['Patient'].y = torch.tensor([G.nodes[pid]['y'] for pid in p_ids], dtype=torch.long)
    data['Patient'].train_mask = torch.tensor([G.nodes[pid]['train_mask'] for pid in p_ids], dtype=torch.bool)
    data['Patient'].val_mask = torch.tensor([G.nodes[pid]['val_mask'] for pid in p_ids], dtype=torch.bool)
    data['Patient'].test_mask = torch.tensor([G.nodes[pid]['test_mask'] for pid in p_ids], dtype=torch.bool)

    # 3. Process ALL Node Types (Initialize x for all types)
    for n_type, mapping in node_mappings.items():
        if n_type == 'Patient':
            continue  # Already handled above
            
        num_nodes = len(mapping)
        data[n_type].num_nodes = num_nodes
        
        # Initialize features to match Patient feature dimension: use torch.zeros or torch.randn. 
        data[n_type].x = torch.zeros((num_nodes, feature_dim), dtype=torch.float)
        
        #print(f"Initialized {n_type} nodes: {num_nodes} nodes with feature dim {feature_dim}")

    # 4. Process Edges
    edge_stores = {} # {(src_type, rel, dst_type): [[src_idx, dst_idx], ...]}

    for u, v, rel, attrs in G.edges(keys=True, data=True):
        u_type = G.nodes[u].get('type')
        v_type = G.nodes[v].get('type')
        
        # Replace double underscores with single ones to satisfy PyG requirements
        safe_rel = str(rel).replace('__', '_')
        
        edge_type = (u_type, safe_rel, v_type)
        
        if edge_type not in edge_stores:
            edge_stores[edge_type] = []
            
        u_idx = node_mappings[u_type][u]
        v_idx = node_mappings[v_type][v]
        
        edge_stores[edge_type].append([u_idx, v_idx])

    # Finalize Edges in HeteroData
    for etype, content in edge_stores.items():
        data[etype].edge_index = torch.tensor(content, dtype=torch.long).t().contiguous()
    
    print(f"HeteroData created: {len(data.node_types)} node types, {len(data.edge_types)} edge types.")
    return data, node_mappings

In [111]:
adni_ecdf_data, adni_ecdf_nodemappings = convert_to_hetero_data(reversed_adni_ecdf)

Starting conversion from NetworkX to HeteroData...
HeteroData created: 25 node types, 827 edge types.


In [113]:
adni_ecdf_data

HeteroData(
  Patient={
    x=[455, 19100],
    y=[455],
    train_mask=[455],
    val_mask=[455],
    test_mask=[455],
  },
  Gene={
    num_nodes=808,
    x=[808, 19100],
  },
  Abundance={
    num_nodes=740,
    x=[740, 19100],
  },
  BiologicalProcess={
    num_nodes=811,
    x=[811, 19100],
  },
  Activity={
    num_nodes=763,
    x=[763, 19100],
  },
  Pathology={
    num_nodes=126,
    x=[126, 19100],
  },
  MicroRna={
    num_nodes=46,
    x=[46, 19100],
  },
  Protein={
    num_nodes=2245,
    x=[2245, 19100],
  },
  Rna={
    num_nodes=768,
    x=[768, 19100],
  },
  Translocation={
    num_nodes=171,
    x=[171, 19100],
  },
  Reaction={
    num_nodes=17,
    x=[17, 19100],
  },
  Degradation={
    num_nodes=79,
    x=[79, 19100],
  },
  CellSecretion={
    num_nodes=72,
    x=[72, 19100],
  },
  CellSurfaceExpression={
    num_nodes=25,
    x=[25, 19100],
  },
  Complex={
    num_nodes=672,
    x=[672, 19100],
  },
  Composite={
    num_nodes=210,
    x=[210, 19100],
  },
 

In [112]:
geo_ecdf_data, geo_ecdf_nodemappings = convert_to_hetero_data(reversed_geo_ecdf)
geo_logfc_data, geo_logfc_nodemappings = convert_to_hetero_data(reversed_geo_logfc)

Starting conversion from NetworkX to HeteroData...
HeteroData created: 25 node types, 827 edge types.
Starting conversion from NetworkX to HeteroData...
HeteroData created: 25 node types, 827 edge types.
